In [1]:
# cell 1 - setup
import os, sys, glob, time, warnings; warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception as e: print("Drive not mounted (ok if files are in /content):", e)
import numpy as np, pandas as pd, lightgbm as lgb
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
SMOKE = os.environ.get("SMOKE") == "1"                   # SMOKE=1 -> tiny quick test of the code path only
def find(name, roots=("/content", "/content/drive/MyDrive", os.getcwd())):
    for r in roots:
        for pat in (name, "*/"+name, "*/*/"+name, "*/*/*/"+name, "*/*/*/*/"+name):
            hit = glob.glob(os.path.join(r, pat))
            if hit: return hit[0]
    return None
TRAIN_PATH, TEST_PATH, SAMPLE_PATH = find("train.csv"), find("test.csv"), find("sample_submission.csv")
assert TRAIN_PATH and TEST_PATH and SAMPLE_PATH, "Could not find train.csv / test.csv / sample_submission.csv - upload them to /content or put them in Drive"
PREV_PATH = find("GCI_submission_qmodels_only*.csv"); HGB_PATH = find("GCI_hgb_product_test.npy")
base_dir = "/content/drive/MyDrive" if os.path.isdir("/content/drive/MyDrive") else "/content"
CACHE_DIR = os.path.join(base_dir, "gci_cache_product") if not SMOKE else "/tmp/gci_cache_smoke"; OUT_DIR = os.path.join(base_dir, "gci_output") if not SMOKE else "/tmp/gci_out_smoke"
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
THREADS = os.cpu_count() or 1
print("train:", TRAIN_PATH, "\ntest:", TEST_PATH, "\nsample:", SAMPLE_PATH, "\nprev q-models:", PREV_PATH, "\nhgb npy:", HGB_PATH, "\ncache:", CACHE_DIR, "| threads:", THREADS)

Mounted at /content/drive
train: /content/train.csv 
test: /content/test.csv 
sample: /content/sample_submission.csv 
prev q-models: None 
hgb npy: None 
cache: /content/drive/MyDrive/gci_cache_product | threads: 2


In [2]:
# cell 2 - features, q and decoded-product keys
CATS = ["NAME_CONTRACT_TYPE","CODE_GENDER","NAME_INCOME_TYPE","NAME_EDUCATION_TYPE","NAME_FAMILY_STATUS","NAME_HOUSING_TYPE","OCCUPATION_TYPE","ORGANIZATION_TYPE"]
tr = pd.read_csv(TRAIN_PATH); te = pd.read_csv(TEST_PATH); sample = pd.read_csv(SAMPLE_PATH)
y = tr["TARGET"].values; ntr = len(tr)
al = pd.concat([tr.drop(columns=["TARGET"]), te], ignore_index=True)
def new_cols(d): return pd.DataFrame(d, index=al.index)
def base(al):
    X = al.drop(columns=["SK_ID_CURR"]).copy()
    for c in CATS: X[c] = X[c].astype("category").cat.codes.replace(-1, np.nan).astype(float)
    return X
def f_clean(al, X):
    d = {"DE_SENT": (al.DAYS_EMPLOYED == 365243).astype(int), "DAYS_EMPLOYED_CLEAN": al.DAYS_EMPLOYED.where(al.DAYS_EMPLOYED != 365243),
         "ORG_XNA": (al.ORGANIZATION_TYPE == "XNA").astype(int), "CAR_AGE_OUT": (al.OWN_CAR_AGE >= 60).astype(int), "MISS_CNT": al.isna().sum(axis=1)}
    for c in ["EXT_SOURCE_1","EXT_SOURCE_3","OWN_CAR_AGE","AMT_REQ_CREDIT_BUREAU_YEAR"]: d[c + "_NA"] = al[c].isna().astype(int)
    return pd.concat([X, new_cols(d)], axis=1)
def f_ext(al, X):
    e = al[["EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3"]]; d = {}
    d["EXT_MEAN"] = e.mean(axis=1); d["EXT_MIN"] = e.min(axis=1); d["EXT_MAX"] = e.max(axis=1); d["EXT_STD"] = e.std(axis=1)
    d["EXT_RANGE"] = d["EXT_MAX"] - d["EXT_MIN"]; d["EXT_NOBS"] = e.notna().sum(axis=1); d["EXT_PROD"] = e.prod(axis=1)
    d["EXT_W"] = (al.EXT_SOURCE_1*2 + al.EXT_SOURCE_2*3 + al.EXT_SOURCE_3*4) / 9
    d["E12"] = al.EXT_SOURCE_1*al.EXT_SOURCE_2; d["E13"] = al.EXT_SOURCE_1*al.EXT_SOURCE_3; d["E23"] = al.EXT_SOURCE_2*al.EXT_SOURCE_3
    d["D12"] = al.EXT_SOURCE_1-al.EXT_SOURCE_2; d["D13"] = al.EXT_SOURCE_1-al.EXT_SOURCE_3; d["D23"] = al.EXT_SOURCE_2-al.EXT_SOURCE_3
    return pd.concat([X, new_cols(d)], axis=1)
def f_fin(al, X):
    inc, cr, an, gp = al.AMT_INCOME_TOTAL, al.AMT_CREDIT, al.AMT_ANNUITY, al.AMT_GOODS_PRICE
    d = {"CREDIT_INC": cr/inc, "ANN_INC": an/inc, "ANN_CREDIT": an/cr, "TERM": cr/an, "GOODS_CREDIT": gp/cr, "CREDIT_MINUS_GOODS": cr-gp, "GOODS_INC": gp/inc,
         "INC_PER_FAM": inc/al.CNT_FAM_MEMBERS, "CREDIT_PER_FAM": cr/al.CNT_FAM_MEMBERS, "ANN_PER_FAM": an/al.CNT_FAM_MEMBERS}
    for c in ["AMT_INCOME_TOTAL","AMT_CREDIT","AMT_ANNUITY","AMT_GOODS_PRICE"]: d["LOG_" + c] = np.log1p(al[c])
    return pd.concat([X, new_cols(d)], axis=1)
def f_age(al, X):
    age = -al.DAYS_BIRTH; de = al.DAYS_EMPLOYED.where(al.DAYS_EMPLOYED != 365243)
    d = {"AGE": age/365.25, "EMP_AGE": de/al.DAYS_BIRTH, "EMP_YEARS": -de/365.25, "REG_AGE": al.DAYS_REGISTRATION/al.DAYS_BIRTH, "ID_AGE": al.DAYS_ID_PUBLISH/al.DAYS_BIRTH,
         "PHONE_AGE": al.DAYS_LAST_PHONE_CHANGE/al.DAYS_BIRTH, "CAR_AGE_RATIO": al.OWN_CAR_AGE/(age/365.25), "CREDIT_PER_AGE": al.AMT_CREDIT/(age/365.25), "INC_PER_AGE": al.AMT_INCOME_TOTAL/(age/365.25)}
    return pd.concat([X, new_cols(d)], axis=1)
def f_bureau(al, X):
    b = [c for c in al.columns if c.startswith("AMT_REQ")]
    d = {"BUR_SUM": al[b].sum(axis=1, min_count=1), "BUR_RECENT": al[["AMT_REQ_CREDIT_BUREAU_HOUR","AMT_REQ_CREDIT_BUREAU_MON"]].sum(axis=1, min_count=1),
         "BUR_MON_YEAR": al.AMT_REQ_CREDIT_BUREAU_MON/(al.AMT_REQ_CREDIT_BUREAU_YEAR+1)}
    return pd.concat([X, new_cols(d)], axis=1)
def f_count(al, X):
    d = {"CNT_" + c: al[c].map(al[c].value_counts()) for c in CATS + ["REGION_POPULATION_RELATIVE","DAYS_BIRTH","AMT_CREDIT","AMT_GOODS_PRICE","AMT_ANNUITY","AMT_INCOME_TOTAL"]}
    return pd.concat([X, new_cols(d)], axis=1)
def f_group(al, X):
    tmp = al.copy(); tmp["CREDIT_INC"] = al.AMT_CREDIT/al.AMT_INCOME_TOTAL; d = {}
    for k in ["ORGANIZATION_TYPE","OCCUPATION_TYPE","NAME_INCOME_TYPE","NAME_EDUCATION_TYPE","REGION_POPULATION_RELATIVE"]:
        for v in ["AMT_INCOME_TOTAL","AMT_CREDIT","EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3","DAYS_BIRTH","CREDIT_INC"]:
            d[f"G_{k[:6]}_{v[-6:]}_DIFF"] = tmp[v] - tmp.groupby(k)[v].transform("mean")
    return pd.concat([X, new_cols(d)], axis=1)
FAMS = {"clean": f_clean, "ext": f_ext, "fin": f_fin, "age": f_age, "bureau": f_bureau, "count": f_count, "group": f_group}
def build(fams):
    X = base(al)
    for f in fams: X = FAMS[f](al, X)
    X = X.replace([np.inf, -np.inf], np.nan)
    return X.iloc[:ntr].reset_index(drop=True), X.iloc[ntr:].reset_index(drop=True)
Xc, Xct = build(["clean", "fin", "age", "bureau"]); Xa, Xat = build(["clean", "ext", "fin", "age", "bureau", "count", "group"])
if SMOKE: Xc, Xa, y = Xc.iloc[:8000], Xa.iloc[:8000], y[:8000]
n, nt = len(y), len(Xct); tidx = np.arange(ntr, ntr + nt) if not SMOKE else np.arange(8000, 8000 + nt)
# ---- q, and the decoded product: annuity = Q0*credit + F  (F ~ 129.5 monthly fee on ~75% of cash loans); label-free ----
q = al.AMT_ANNUITY / al.AMT_CREDIT; _cash = (al.NAME_CONTRACT_TYPE == "Cash loans").values; _qb = (al.AMT_ANNUITY - 129.5) / al.AMT_CREDIT
_q5, _qb5 = q.round(5), _qb.round(5)
FEE = ((_qb5.map(_qb5.value_counts()).values > _q5.map(_q5.value_counts()).values) & _cash).astype(int); Q0 = pd.Series(np.where(FEE, _qb, q))
KS = {"Q3": q.round(3).astype(str), "Q4": q.round(4).astype(str), "Q5": q.round(5).astype(str),
      "P3": Q0.round(3).astype(str), "P4": Q0.round(4).astype(str), "P5": Q0.round(5).astype(str), "PF4": Q0.round(4).astype(str) + "|" + pd.Series(FEE).astype(str)}
M0 = {"Q3": 100., "Q4": 30., "Q5": 30., "P3": 100., "P4": 30., "P5": 30., "PF4": 30.}
if SMOKE: KS = {k: v.iloc[list(range(8000)) + list(range(ntr, ntr + nt))].reset_index(drop=True) for k, v in KS.items()}
print("decoded products:", Q0.round(5).nunique(), "(q5 distinct:", q.round(5).nunique(), ") | fee share of cash loans:", round(float(FEE[_cash].mean()), 3), "| features core/all:", Xc.shape[1], Xa.shape[1])
def enc(fit, qi, keys=None):
    pr = float(y[fit].mean()); out = {}
    for k in (keys or KS):
        key = KS[k]; g = pd.DataFrame({"g": key.iloc[fit].values, "y": y[fit]}).groupby("g")["y"].agg(["sum", "count"]); s = key.iloc[qi]
        cnt = s.map(g["count"]).fillna(0).to_numpy(float); sm = s.map(g["sum"]).fillna(0).to_numpy(float)
        out[f"TE_{k}"] = (sm + M0[k] * pr) / (cnt + M0[k]); out[f"CNT_{k}"] = cnt
    return pd.DataFrame(out)
def train_enc(a):     # nested out-of-fold encodings for training rows (a row's own label never enters its own feature)
    cols = enc(a[:10], a[:5]).columns; inner = np.zeros((len(a), len(cols)))
    for ia, ib in StratifiedKFold(5, shuffle=True, random_state=7).split(a, y[a]): inner[ib] = enc(a[ia], a[ib]).values
    return pd.DataFrame(inner, columns=cols)
def onehot(Xa_, Xt_):
    z = pd.concat([Xa_, Xt_], ignore_index=True); oh = pd.get_dummies(z[CATS].astype("category"), dummy_na=True, dtype=np.float32)
    oh.columns = [f"OH_{i}" for i in range(oh.shape[1])]; Z = pd.concat([z.drop(columns=CATS), oh], axis=1)
    return Z.iloc[:len(Xa_)].reset_index(drop=True), Z.iloc[len(Xa_):].reset_index(drop=True)
Xco, Xcot = onehot(Xc, Xct); Xao, Xaot = onehot(Xa, Xat)
MONO = [-1 if c in ("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3") else 0 for c in Xc.columns] + [0] * (2 * len(KS))   # + TE/CNT columns of every key
BASE = dict(objective="binary", bagging_freq=1, verbose=-1, num_threads=THREADS, seed=42)
# name: (train feats, test feats, params, rounds for fold models, rounds for full-data models)
MEMBERS = {
 "qf_onehot":    (Xco, Xcot, dict(learning_rate=0.03, num_leaves=11, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20), 1939, 2133),
 "qf_allfam_oh": (Xao, Xaot, dict(learning_rate=0.04, num_leaves=15, min_child_samples=150, feature_fraction=0.3, bagging_fraction=0.8, lambda_l2=20), 881, 969),
 "qf_mono":      (Xc.reset_index(drop=True), Xct.reset_index(drop=True), dict(learning_rate=0.02, num_leaves=15, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20, monotone_constraints=MONO), 1571, 1729)}
REF = {"qf_onehot": 0.76713, "qf_allfam_oh": 0.76678, "qf_mono": 0.76746}      # held-out AUC of the same members WITHOUT product keys (your research.md)
rk = lambda v: (rankdata(v) - 1) / (len(v) - 1)
FOLDS = 2 if SMOKE else 5
def run_member(name):
    X, Xt, p, R, Rf = MEMBERS[name]; f = os.path.join(CACHE_DIR, name + ".npz")
    if os.path.exists(f) and not SMOKE:
        z = np.load(f); print(f"{name}: loaded from cache | held-out AUC {roc_auc_score(y, z['oof']):.5f} (reference without product keys {REF[name]:.5f})"); return z["test"], z["oof"]
    tf = np.zeros(nt); oof = np.zeros(n); t0 = time.time()
    for k, (a, b) in enumerate(StratifiedKFold(FOLDS, shuffle=True, random_state=42).split(np.zeros(n), y)):
        fa = pd.concat([X.iloc[a].reset_index(drop=True), train_enc(a)], axis=1); fb = pd.concat([X.iloc[b].reset_index(drop=True), enc(a, b)], axis=1); ft = pd.concat([Xt.reset_index(drop=True), enc(a, tidx)], axis=1)
        m = lgb.train({**BASE, **p}, lgb.Dataset(fa, y[a]), 60 if SMOKE else R); oof[b] = m.predict(fb); tf += m.predict(ft) / FOLDS
        print(f"  {name} fold {k+1}/{FOLDS}: AUC {roc_auc_score(y[b], oof[b]):.5f} ({time.time()-t0:.0f}s)", flush=True)
    allr = np.arange(n); fa = pd.concat([X.reset_index(drop=True), train_enc(allr)], axis=1); ft = pd.concat([Xt.reset_index(drop=True), enc(allr, tidx)], axis=1); fr = []
    for sd in ([42] if SMOKE else [42, 7, 2024]):
        fr.append(rk(lgb.train({**BASE, **p, "seed": sd}, lgb.Dataset(fa, y), 60 if SMOKE else Rf).predict(ft))); print(f"  {name} full-data seed {sd} done ({time.time()-t0:.0f}s)", flush=True)
    test = rk(0.5 * rk(tf) + 0.5 * np.mean(fr, 0))
    print(f"{name}: held-out AUC {roc_auc_score(y, oof):.5f}  vs reference without product keys {REF[name]:.5f}  -> change {roc_auc_score(y, oof) - REF[name]:+.5f}", flush=True)
    if not SMOKE: np.savez(f, test=test, oof=oof)
    return test, oof
RES = {}

decoded products: 1047 (q5 distinct: 5335 ) | fee share of cash loans: 0.75 | features core/all: 67 130


In [3]:
# cell 3a - member 1 (about 20-35 min on Colab CPU)
RES["qf_onehot"] = run_member("qf_onehot")

  qf_onehot fold 1/5: AUC 0.76804 (71s)
  qf_onehot fold 2/5: AUC 0.77217 (118s)
  qf_onehot fold 3/5: AUC 0.77470 (165s)
  qf_onehot fold 4/5: AUC 0.76703 (234s)
  qf_onehot fold 5/5: AUC 0.76792 (303s)
  qf_onehot full-data seed 42 done (391s)
  qf_onehot full-data seed 7 done (480s)
  qf_onehot full-data seed 2024 done (570s)
qf_onehot: held-out AUC 0.76992  vs reference without product keys 0.76713  -> change +0.00279


In [4]:
# cell 3b - member 2 (about 10-20 min)
RES["qf_allfam_oh"] = run_member("qf_allfam_oh")

  qf_allfam_oh fold 1/5: AUC 0.76748 (30s)
  qf_allfam_oh fold 2/5: AUC 0.77272 (60s)
  qf_allfam_oh fold 3/5: AUC 0.77639 (91s)
  qf_allfam_oh fold 4/5: AUC 0.76842 (121s)
  qf_allfam_oh fold 5/5: AUC 0.76879 (151s)
  qf_allfam_oh full-data seed 42 done (199s)
  qf_allfam_oh full-data seed 7 done (233s)
  qf_allfam_oh full-data seed 2024 done (268s)
qf_allfam_oh: held-out AUC 0.77070  vs reference without product keys 0.76678  -> change +0.00392


In [5]:
# cell 3c - member 3 (about 25-40 min)
RES["qf_mono"] = run_member("qf_mono")

  qf_mono fold 1/5: AUC 0.76771 (46s)
  qf_mono fold 2/5: AUC 0.77201 (90s)
  qf_mono fold 3/5: AUC 0.77543 (135s)
  qf_mono fold 4/5: AUC 0.76822 (179s)
  qf_mono fold 5/5: AUC 0.76864 (227s)
  qf_mono full-data seed 42 done (283s)
  qf_mono full-data seed 7 done (335s)
  qf_mono full-data seed 2024 done (389s)
qf_mono: held-out AUC 0.77035  vs reference without product keys 0.76746  -> change +0.00289


In [6]:
# cell 4 - ensemble check and submission files
oofavg = np.mean([rk(RES[k][1]) for k in MEMBERS], 0); print(f"3-member held-out AUC (rank average): {roc_auc_score(y, oofavg):.5f}   [reference without product keys: 0.76843]")
lgb_test = np.mean([RES[k][0] for k in MEMBERS], 0); ids = sample.SK_ID_CURR
def write(name, vals):
    out = pd.DataFrame({"SK_ID_CURR": ids, "TARGET": vals}); assert len(out) == 61500 and out.SK_ID_CURR.is_unique and np.isfinite(out.TARGET).all()
    p = os.path.join(OUT_DIR, name); out.to_csv(p, index=False); print("wrote", p); return p
files = [write("A_lgb_product_only.csv", lgb_test)]
if HGB_PATH:
    hgb = np.load(HGB_PATH); assert len(hgb) == 61500
    files.append(write("B_lgb70_hgb30.csv", 0.7 * lgb_test + 0.3 * rk(hgb))); files.append(write("C_lgb50_hgb50.csv", 0.5 * lgb_test + 0.5 * rk(hgb)))
if PREV_PATH:
    prev = pd.read_csv(PREV_PATH); assert np.array_equal(prev.SK_ID_CURR.values, ids.values); files.append(write("D_lgb50_prevq50.csv", 0.5 * lgb_test + 0.5 * rk(prev.TARGET.values)))
try:
    from google.colab import files as cf
    for p in files: cf.download(p)
except Exception: print("download manually from", OUT_DIR)

3-member held-out AUC (rank average): 0.77158   [reference without product keys: 0.76843]
wrote /content/drive/MyDrive/gci_output/A_lgb_product_only.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>